# TASK-006 ? Prompt engineering for synthetic AG News

This notebook compares a basic and a diversity-focused prompt using the same local Qwen3 4B Instruct checkpoint, decoding settings, category order, and random seeds. It generates five items per AG News category with each prompt (40 total). Generated texts are *fictional research data*, not verified news.

Run the cells in order from the repository root. The model is the locally installed Ollama `qwen3:4b-instruct` tag (Qwen3-4B-Instruct-2507, Q4_K_M); the notebook records its exact digest. Start Ollama before running. Python dependencies for this notebook are `numpy`, `pandas`, `scikit-learn`, `requests`, and `ipython`. It does not need PyTorch or Transformers for inference.

Sources: [Qwen3 4B model tags](https://ollama.com/library/qwen3/tags), [Ollama API](https://github.com/ollama/ollama/blob/main/docs/api.md), [AG News dataset](https://huggingface.co/datasets/fancyzhx/ag_news).

In [1]:
from pathlib import Path
import hashlib
import itertools
import json
import re
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "prompts" / "basic.txt").exists():
            return candidate
    raise FileNotFoundError("Run Jupyter from the repository or its notebooks directory.")


ROOT = find_project_root()
PROMPTS_DIR = ROOT / "prompts"
DATA_OUT = ROOT / "data" / "synthetic" / "task006"
RESULTS_OUT = ROOT / "results" / "task006"
DATA_OUT.mkdir(parents=True, exist_ok=True)
RESULTS_OUT.mkdir(parents=True, exist_ok=True)

MODEL_ID = "qwen3:4b-instruct"
API_URL = "http://127.0.0.1:11434"
SEED = 42
N_PER_CATEGORY = 5
MAX_NEW_TOKENS = 180

CATEGORIES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
SCENARIOS = {
    "World": [
        "regional diplomatic talks in East Africa",
        "election administration in Southeast Asia",
        "cross-border transport agreement in Europe",
        "flood relief coordination in South America",
        "trade negotiations in the Middle East",
    ],
    "Sports": [
        "regional football final", "tennis qualifying upset",
        "athletics championship selection", "basketball coaching change",
        "cricket tournament opener",
    ],
    "Business": [
        "small manufacturer earnings", "retail chain expansion",
        "labor contract negotiations", "agricultural commodity prices",
        "competition review of a proposed merger",
    ],
    "Sci/Tech": [
        "medical device prototype", "software security update",
        "small satellite mission", "battery materials research",
        "AI tool deployment at a university",
    ],
}
assert all(len(SCENARIOS[category]) == N_PER_CATEGORY for category in CATEGORIES.values())

prompts = {
    "system": "You write original, plausible, fictional English news briefs for a text-classification research dataset. Do not reproduce or claim to report a verified real event. Return only two lines headed exactly 'Headline:' and 'Summary:'. Do not add explanations, markdown, or category labels.",
    "basic": (PROMPTS_DIR / "basic.txt").read_text(encoding="utf-8").strip(),
    "advanced": (PROMPTS_DIR / "advanced.txt").read_text(encoding="utf-8").strip(),
}
prompt_sha256 = hashlib.sha256(
    json.dumps(prompts, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
).hexdigest()
display(pd.DataFrame({"strategy": ["basic", "advanced"],
                      "prompt": [prompts["basic"], prompts["advanced"]]}))

,strategy,prompt
0,basic,Write one short AG News-style news item in the...
1,advanced,Write one original AG News-style news item in ...


## Controlled design

Each strategy uses exactly one generation per category and replicate. The
advanced prompt adds a different scenario for each replicate. Sampling seeds
are paired across strategies, but generations are not expected to be identical.
Both strategies use the same system prompt, format, checkpoint, and decoding.
The AG News labels are 0 World, 1 Sports, 2 Business, 3 Sci/Tech.

In [2]:
plan = pd.DataFrame([
    {
        "sample_id": f"{strategy}_{label}_{replicate + 1}",
        "strategy": strategy,
        "label": label,
        "category": category,
        "replicate": replicate + 1,
        "seed": SEED + label * 100 + replicate,
        "scenario": SCENARIOS[category][replicate] if strategy == "advanced" else None,
    }
    for strategy, (label, category), replicate in itertools.product(
        ("basic", "advanced"), CATEGORIES.items(), range(N_PER_CATEGORY)
    )
])
assert len(plan) == 40 and not plan.sample_id.duplicated().any()
display(plan.groupby(["strategy", "category"]).size().unstack())

category,Business,Sci/Tech,Sports,World
strategy,,,,
advanced,5,5,5,5
basic,5,5,5,5


## Connect to local Qwen3 4B

The installed Ollama model is a 4-bit quantized Qwen3-4B-Instruct-2507 checkpoint. Ollama runs it locally on the available hardware. The API model digest is saved in the experiment manifest. Non-thinking mode and the same decoding options are used for both strategies.

In [3]:
import requests

session = requests.Session()
tags_response = session.get(f"{API_URL}/api/tags", timeout=10)
tags_response.raise_for_status()
model_record = next(
    (item for item in tags_response.json()["models"] if item["name"] == MODEL_ID), None
)
if model_record is None:
    raise RuntimeError(f"Install the local model first: ollama pull {MODEL_ID}")
MODEL_DIGEST = model_record["digest"]
OLLAMA_VERSION = session.get(f"{API_URL}/api/version", timeout=10).json().get("version")
print({"model": MODEL_ID, "digest": MODEL_DIGEST,
       "quantization": model_record.get("details", {}).get("quantization_level"),
       "ollama_version": OLLAMA_VERSION})

{'model': 'qwen3:4b-instruct', 'digest': '0edcdef34593eac1aa2be9c7d06c432dcf81945adca5eca2f27662c18f168ba0', 'quantization': 'Q4_K_M', 'ollama_version': '0.40.11'}


## Generate and checkpoint all 40 samples

Each completed item is appended to JSONL, so interrupting the notebook does
not lose previous generations. A manifest prevents accidental mixing of runs
with different prompts or decoding settings. Raw output is retained even when
the requested format is not followed; invalid items are not silently removed.

In [4]:
manifest_path = RESULTS_OUT / "manifest.json"
samples_path = DATA_OUT / "samples.jsonl"
manifest = {
    "backend": "Ollama",
    "model_id": MODEL_ID,
    "model_digest": MODEL_DIGEST,
    "model_variant": "Qwen3-4B-Instruct-2507 Q4_K_M",
    "ollama_version": OLLAMA_VERSION,
    "prompt_sha256": prompt_sha256,
    "prompt_files": ["prompts/basic.txt", "prompts/advanced.txt"],
    "seed": SEED,
    "n_per_category": N_PER_CATEGORY,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "think": False,
}
if manifest_path.exists():
    previous = json.loads(manifest_path.read_text(encoding="utf-8"))
    if previous != manifest:
        raise RuntimeError("Existing TASK-006 output uses different settings. Move it before a new run.")
else:
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")


def parse_article(raw: str) -> tuple[str, str, bool]:
    match = re.fullmatch(r"\s*Headline:\s*([^\n]+)\s*\nSummary:\s*(.+?)\s*",
                         raw, flags=re.IGNORECASE | re.DOTALL)
    if match is None:
        return "", "", False
    return match.group(1).strip(), match.group(2).strip(), True


def generate_one(item: dict) -> dict:
    content = prompts[item["strategy"]].format(
        category=item["category"], scenario=item["scenario"]
    )
    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "system", "content": prompts["system"]},
            {"role": "user", "content": content},
        ],
        "stream": False,
        "think": False,
        "options": {
            "seed": item["seed"], "num_predict": MAX_NEW_TOKENS,
            "temperature": 0.7, "top_p": 0.8, "top_k": 20,
        },
    }
    start = time.perf_counter()
    response = session.post(f"{API_URL}/api/chat", json=payload, timeout=300)
    response.raise_for_status()
    result = response.json()
    elapsed = time.perf_counter() - start
    raw = result["message"]["content"].strip()
    headline, summary, valid = parse_article(raw)
    return {
        **item, "prompt": content, "raw_output": raw,
        "headline": headline, "summary": summary,
        "text": f"{headline} {summary}".strip() if valid else raw,
        "format_valid": valid, "new_tokens": result.get("eval_count"),
        "seconds": round(elapsed, 3), "done_reason": result.get("done_reason"),
    }


existing = []
if samples_path.exists():
    existing = [json.loads(line) for line in samples_path.read_text(encoding="utf-8").splitlines() if line]
done = {row["sample_id"] for row in existing}
if len(done) != len(existing) or not done.issubset(set(plan.sample_id)):
    raise RuntimeError("Existing samples contain duplicate or unexpected IDs.")

for item in plan.to_dict("records"):
    if item["sample_id"] in done:
        continue
    result = generate_one(item)
    with samples_path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")
    done.add(item["sample_id"])
    print(f"{len(done):02d}/40 {item['sample_id']} ? {result['seconds']:.1f}s; format={result['format_valid']}")

samples = pd.read_json(samples_path, lines=True)
assert len(samples) == 40 and not samples.sample_id.duplicated().any()
display(samples[["sample_id", "category", "headline", "summary", "format_valid"]].head())

,sample_id,category,headline,summary,format_valid
0,basic_0_1,World,Global Climate Summit Ends with Historic Agree...,Leaders from 195 countries reached a landmark ...,True
1,basic_0_2,World,UN Calls for Global Action on Climate-Induced ...,The United Nations has urged nations to streng...,True
2,basic_0_3,World,UN Warns of Climate Threat to Global Food Secu...,The United Nations has issued a stark warning ...,True
3,basic_0_4,World,Global Leaders Convene in Geneva to Address Cl...,Diplomats from 120 countries gathered in Genev...,True
4,basic_0_5,World,UN Appeals for Global Aid as Sahel Drought Wor...,A severe drought in the Sahel region has displ...,True


## Automatic quality and diversity checks

Format and length measure instruction following, not factual accuracy or
realism. `distinct_2` is the proportion of unique word bigrams in each five
item category group. Mean pairwise TF-IDF cosine similarity is a rough lexical
similarity measure (lower may indicate more diversity). A second pass uses only the first 35 words of each item to reduce
length confounding. With only five items
per group, these are descriptive statistics, not significance tests.

In [5]:
WORD = re.compile(r"[^\W\d_]+(?:'[^\W\d_]+)?", flags=re.UNICODE)


def words(text: str) -> list[str]:
    return WORD.findall(text.lower())


def distinct_2(texts: list[str]) -> float:
    bigrams = [pair for text in texts for pair in zip(words(text), words(text)[1:])]
    return len(set(bigrams)) / len(bigrams) if bigrams else float("nan")


def mean_pairwise_similarity(texts: list[str]) -> float:
    if len(texts) < 2 or not any(text.strip() for text in texts):
        return float("nan")
    try:
        matrix = TfidfVectorizer(stop_words="english").fit_transform(texts)
    except ValueError:  # all texts may contain only stop words or punctuation
        return float("nan")
    sims = cosine_similarity(matrix)
    return float(sims[np.triu_indices(len(texts), k=1)].mean())


samples["word_count"] = samples.text.map(lambda text: len(words(text)))
samples["length_valid"] = samples.word_count.between(35, 80)
samples["exact_duplicate"] = samples.duplicated(["strategy", "category", "text"], keep=False)
samples.to_csv(RESULTS_OUT / "sample_metrics.csv", index=False, encoding="utf-8")

comparison = (
    samples.groupby(["strategy", "category"], sort=False)
    .agg(n=("sample_id", "size"), format_rate=("format_valid", "mean"),
         length_rate=("length_valid", "mean"), mean_words=("word_count", "mean"),
         duplicate_rate=("exact_duplicate", "mean"), mean_seconds=("seconds", "mean"))
    .reset_index()
)
diversity = samples.groupby(["strategy", "category"], sort=False).text.apply(list).reset_index()
diversity["distinct_2"] = diversity.text.map(distinct_2)
diversity["mean_tfidf_cosine"] = diversity.text.map(mean_pairwise_similarity)
first_35 = diversity.text.map(lambda texts: [" ".join(words(text)[:35]) for text in texts])
diversity["distinct_2_first35"] = first_35.map(distinct_2)
diversity["mean_tfidf_cosine_first35"] = first_35.map(mean_pairwise_similarity)
comparison = comparison.merge(diversity.drop(columns="text"), on=["strategy", "category"])
comparison.to_csv(RESULTS_OUT / "comparison.csv", index=False, encoding="utf-8")
display(comparison.round(3))

,strategy,category,n,format_rate,length_rate,mean_words,duplicate_rate,mean_seconds,distinct_2,mean_tfidf_cosine,distinct_2_first35,mean_tfidf_cosine_first35
0,basic,World,5,1.0,1.0,49.2,0.0,1.943,0.917,0.103,0.935,0.092
1,basic,Sports,5,1.0,1.0,49.2,0.0,1.340,0.855,0.098,0.835,0.111
2,basic,Business,5,1.0,1.0,49.4,0.0,1.228,0.769,0.220,0.706,0.261
3,basic,Sci/Tech,5,1.0,1.0,54.4,0.0,1.216,0.768,0.193,0.688,0.228
4,advanced,World,5,1.0,1.0,63.6,0.0,1.495,0.946,0.037,0.941,0.017
5,advanced,Sports,5,1.0,0.8,70.8,0.0,2.023,0.917,0.019,0.912,0.017
6,advanced,Business,5,1.0,1.0,62.0,0.0,1.544,0.951,0.033,0.941,0.011
7,advanced,Sci/Tech,5,1.0,1.0,65.0,0.0,1.605,0.941,0.040,0.924,0.029


## Pilot result and qualitative audit

The completed local run used the Qwen3-4B-Instruct-2507 Q4_K_M checkpoint recorded in `manifest.json`. Both strategies produced 20/20 correctly formatted items and no exact duplicates. All 20 basic items and 19/20 advanced items met the 35?80 word target; `advanced_1_2` has 84 words.

| Strategy | Mean distinct-2, first 35 words | Mean within-category TF-IDF cosine, first 35 words |
|---|---:|---:|
| Basic | 0.791 | 0.173 |
| Advanced | 0.929 | 0.018 |

These are averages of four category-level scores. The advanced strategy is more lexically varied even after matching the first 35 words. This is a small descriptive pilot, not evidence of a statistically significant improvement or better downstream classifier performance.

Example successes and failures:

- The basic Business set names Apple in all five items, and three of five basic Sci/Tech items use protein folding. The scenario-guided advanced set covers five different topics in each category.
- `advanced_0_3` describes freight rail access through the Mont Blanc tunnel under a France?Switzerland agreement. The [tunnel operator](https://www.atmb.com/connaitre-atmb/qui-est-atmb/le-tunnel-du-mont-blanc-ouvrage-exceptionnel-innovant-et-securise/) describes a road tunnel connecting France and Italy. This is a concrete realism failure despite greater diversity.
- `basic_2_1` and `basic_2_2` present different invented Apple investment amounts for almost the same story. Such near-repetition and conflicting specifics could bias or confuse a classifier.

The generated briefs can name real organizations while describing fictional events. Format and lexical metrics do not establish factual plausibility or category fit. The unscored review sheet below is the remaining quality check before training with these items.

## Human quality review

Review each item for **category fit** and **realism** on a 1–5 scale. Check
concrete errors, unsupported specificity, repetition, and label ambiguity in
the notes. Enter scores in `results/task006/review_template.csv`, then
rerun the cell below. A prompt's quality cannot be inferred from format alone.

In [6]:
review_path = RESULTS_OUT / "review_template.csv"
if not review_path.exists():
    review = samples[["sample_id", "strategy", "category", "headline", "summary"]].copy()
    review["category_fit_1to5"] = pd.NA
    review["realism_1to5"] = pd.NA
    review["notes"] = ""
    review.to_csv(review_path, index=False, encoding="utf-8")

review = pd.read_csv(review_path)
for column in ("category_fit_1to5", "realism_1to5"):
    review[column] = pd.to_numeric(review[column], errors="coerce")
    if review[column].notna().any() and not review[column].dropna().between(1, 5).all():
        raise ValueError(f"{column} must contain only scores from 1 to 5")

reviewed = review.dropna(subset=["category_fit_1to5", "realism_1to5"])
if reviewed.empty:
    print(f"Review template saved to {review_path}; human scores are pending.")
else:
    display(reviewed.groupby(["strategy", "category"])[
        ["category_fit_1to5", "realism_1to5"]
    ].agg(["count", "mean"]).round(2))
    if len(reviewed) != len(samples):
        print(f"Partial review: {len(reviewed)}/{len(samples)} items scored.")

Review template saved to D:\Projects\5_sem\Natural Language Processing\lab4\Synthetic-Data-NLP_Lab-4_3rd-course-\results\task006\review_template.csv; human scores are pending.


## Interpretation checklist

- Compare basic and advanced within each category, especially duplicate rate,
  distinct-2, and mean similarity; inspect the actual examples before claiming
  that diversity improved.
- Compare format and length rates; any diversity gain may carry a quality cost.
- After human scoring, report category fit and realism, with representative
  successes and failures from both prompts.
- This pilot has only five items per category and no independent fact check.
  Do not draw strong statistical conclusions from its descriptive metrics.